## Titanic 生存预测
同济大学土木水利研一 | 2026年5月

## 项目目标
用机器学习预测泰坦尼克号乘客的生存情况,最终提交 Kaggle 得分 0.78947

## 流程
1. 读取数据
2. 探索性数据分析
3. 特征工程
4. 训练模型
5. 生成提交文件

In [1]:
# ===== 1. 读取数据 =====
import pandas as pd
import numpy as np

train = pd.read_csv('train.csv') # 训练集：891行，有Survived标签
test = pd.read_csv('test.csv') # 测试集：418行，需要预测

print(train.shape)
print(train.head()) #默认显示前五行

(891, 12)
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN    

In [2]:
# ===== 2. 了解数据缺失情况 =====
print("=== 缺失值统计 ===") 
print(train.isnull().sum())# isnull() 检查每个格子是否为空，返回True/False# sum() 把True当1加起来，得到缺失值数量

# ！！！value_counts(normalize=True)——转成比例（如果值不为0或者1））！！！
# Survived只有0和1，mean()直接算出存活率
# :.2% 是保留2位小数显示成百分比
print("\n=== 存活率 ===")
print(train['Survived'].value_counts())
print(f"存活率：{train['Survived'].mean():.2%}")

=== 缺失值统计 ===
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

=== 存活率 ===
Survived
0    549
1    342
Name: count, dtype: int64
存活率：38.38%


In [3]:
# ===== 3. 特征工程 =====
all_data = pd.concat([train,test], ignore_index=True) #contact合并训练集和测试集便于操作，改变行号
print(all_data.shape)

(1309, 12)


In [4]:
print(test['Embarked'].unique()) #看一下Embarked列的特征值

['Q' 'S' 'C']


In [5]:
all_data['Age'] = all_data['Age'].fillna(all_data['Age'].median()) #把Age缺失的空位用中位数补上（因为年龄有极端情况，中位数合适一点）
all_data['Embarked'] = all_data['Embarked'].fillna(all_data['Embarked'].mode()[0]) #同理，Embarked登船地点用众数补上，概率大一点
print(all_data[['Age','Embarked']].isnull().sum()) #检查一下这两列还有没有缺失值了。Tips：取多列要用两个括号，括号里面加列表

Age         0
Embarked    0
dtype: int64


In [6]:
all_data['Has_Cabin'] = all_data['Cabin'].notnull().astype(int) #（这步有点繁琐了其实不需要新建列的，不过这样会看得清晰一点）notnull是不为空，有值=True反之亦然，再加上astype（int）转换成0/1，整行的意思是有船舱号的标记为1，没有的标记为0，存到新列 Has_Cabin 里
print(all_data['Has_Cabin'])

0       0
1       1
2       0
3       1
4       0
       ..
1304    0
1305    1
1306    0
1307    0
1308    0
Name: Has_Cabin, Length: 1309, dtype: int64


In [7]:
all_data['Sex'] = all_data['Sex'].map({'female':1, 'male':0}) #把Sex列的female和male变为数字，让模型识别预测
print(all_data['Sex'])

0       0
1       1
2       1
3       1
4       0
       ..
1304    0
1305    1
1306    0
1307    0
1308    0
Name: Sex, Length: 1309, dtype: int64


In [8]:
print(all_data['Sex'].unique()) #看Sex现在的特征值是什么

[0 1]


In [9]:
all_data['Embarked'] = all_data['Embarked'].map({'S':0, 'C':1, 'Q':2}) #同理，换成数字
print(all_data['Embarked'])

0       0
1       1
2       0
3       0
4       0
       ..
1304    0
1305    1
1306    0
1307    0
1308    1
Name: Embarked, Length: 1309, dtype: int64


In [10]:
all_data = all_data.drop(['Name','Ticket','Cabin','PassengerId'],axis=1) #去除四列对预测没影响的
print(all_data)

      Survived  Pclass  Sex   Age  SibSp  Parch      Fare  Embarked  Has_Cabin
0          0.0       3    0  22.0      1      0    7.2500         0          0
1          1.0       1    1  38.0      1      0   71.2833         1          1
2          1.0       3    1  26.0      0      0    7.9250         0          0
3          1.0       1    1  35.0      1      0   53.1000         0          1
4          0.0       3    0  35.0      0      0    8.0500         0          0
...        ...     ...  ...   ...    ...    ...       ...       ...        ...
1304       NaN       3    0  28.0      0      0    8.0500         0          0
1305       NaN       1    1  39.0      0      0  108.9000         1          1
1306       NaN       3    0  38.5      0      0    7.2500         0          0
1307       NaN       3    0  28.0      0      0    8.0500         0          0
1308       NaN       3    0  28.0      1      1   22.3583         1          0

[1309 rows x 9 columns]


In [11]:
 #分开训练集和测试集，呼应上文，毕竟训练集是用来训练模型的，测试集使我们要用来预测的
train_new = all_data[:891]
test_new = all_data[891:]

In [12]:
print(train_new.shape)
print(test_new.shape)

(891, 9)
(418, 9)


In [13]:
#x（小写哦，x 是输入，y 是输出，机器学习借用了数学这个习惯）需要删Survived这一列
x = train_new.drop(['Survived'],axis=1) 
y = train_new['Survived']

In [14]:
# ===== 4. 训练模型 =====
from sklearn.ensemble import RandomForestClassifier  # 从sklearn工具箱里导入逻辑回归算法

model = RandomForestClassifier(n_estimators=150,max_depth=6,random_state=42) 
model.fit(x, y)  # 用训练集数据训练模型，x是特征，y是答案

RandomForestClassifier(max_depth=6, n_estimators=150, random_state=42)

In [15]:
test_new = test_new.copy() #把test_new变成独立副本，避免修改原数据时报警告
test_new['Fare'] = test_new['Fare'].fillna(test_new['Fare'].median()) #测试集Fare列缺1个值，用中位数填充(输出结果的时候Fare报错了，之前就应该看一下test情况）
predictions = model.predict(test_new.drop(['Survived'],axis=1)) #删掉Survived列（全是NaN），用模型预测测试集

In [16]:
print(predictions)

[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0. 1. 0.
 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0.
 1. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 1. 1. 1. 0. 0. 1. 1. 0.
 0. 0. 1. 0. 0. 1. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 1. 1. 0. 0. 1. 0. 0. 0.
 1. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 0. 0. 1. 0. 1.
 1. 0. 1. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.
 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 1. 1. 0. 1. 1. 1. 1. 0. 0. 0. 0. 0.
 1. 0. 0. 0. 0. 0. 0. 1. 1. 0. 1. 1. 0. 1. 1. 0. 1. 0. 1. 0. 0. 0. 0. 0.
 0. 0. 1. 0. 1. 1. 0. 0. 1. 1. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0.
 1. 0. 1. 0. 1. 0. 1. 0. 1. 1. 0. 1. 0. 0. 0. 1. 0. 0. 1. 0. 0. 0. 1. 1.
 1. 1. 0. 0. 0. 0. 1. 0. 1. 0. 1. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 1.
 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 1. 0. 0. 0. 0. 0. 1. 1. 1. 1. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 1. 0. 0. 0. 0.
 0. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 1. 0. 0.

In [17]:
# ===== 5. 生成提交文件 =====
submission = pd.read_csv('gender_submission.csv') #读取Kaggle的提取模版
submission['Survived'] = predictions.astype(int) #把预测结果填入Survived列，astype(int)确保是整数0或1
submission.to_csv('submission.csv',index=False) #保存成CSV文件，index=False不把行号写进文件（不然会报错）
print('提交文件生成成功！')

提交文件生成成功！
